In [ ]:
import os
from bson import ObjectId
from dotenv import load_dotenv
from pymongo import MongoClient

load_dotenv()  # carrega .env da raiz

host = os.getenv("MONGODB_HOST")
db_name = os.getenv("MONGODB_DATABASE")
user = os.getenv("MONGODB_USER")
password = os.getenv("MONGODB_PASSWORD")
auth_collection = os.getenv("MONGODB_AUTH_COLLECTION")

uri = f"mongodb://{user}:{password}@{host}:27017/{db_name}"

client = MongoClient(uri)
db = client[db_name]

# teste rápido
print(db.list_collection_names())


In [ ]:
for doc in db[auth_collection].find({}, {"_id": 0}):
    print(doc)

### Adicionar usuário

In [ ]:
import os
from urllib.parse import quote_plus

import ipywidgets as widgets
from IPython.display import display, clear_output
from pymongo import MongoClient, errors
from streamlit_authenticator.utilities.hasher import Hasher

# Mongo config
user = os.environ["MONGODB_USER"]
pw = os.environ["MONGODB_PASSWORD"]
host = os.environ["MONGODB_HOST"]
db = os.environ["MONGODB_DATABASE"]
auth_collection = os.getenv("MONGODB_AUTH_COLLECTION")

uri = f"mongodb://{quote_plus(user)}:{quote_plus(pw)}@{host}/{db}"
client = MongoClient(uri)
collection = client[db][auth_collection]

# Garante unicidade
collection.create_index("username", unique=True)
collection.create_index("email", unique=True)

# Widgets
w_username = widgets.Text(description="Username")
w_email = widgets.Text(description="Email")
w_first = widgets.Text(description="Nome")
w_last = widgets.Text(description="Sobrenome")
w_password = widgets.Password(description="Senha")
w_roles = widgets.SelectMultiple(
    options=["user", "admin", "viewer", "editor"],
    value=["user"],
    description="Roles",
)
w_button = widgets.Button(description="Registrar", button_style="success")
w_out = widgets.Output()

def on_register(_):
    with w_out:
        clear_output()
        username = w_username.value.strip().lower()
        email = w_email.value.strip().lower()
        first = w_first.value.strip()
        last = w_last.value.strip()
        password = w_password.value

        if not username or not email or not password:
            print("Preencha username, email e senha.")
            return
        if "@" not in email:
            print("Email inválido.")
            return

        try:
            collection.insert_one({
                "username": username,
                "email": email,
                "first_name": first,
                "last_name": last,
                "password": Hasher.hash(password),
                "roles": list(w_roles.value),
                "failed_login_attempts": 0,
                "logged_in": False,
            })
            print("Usuário criado com sucesso.")
            w_password.value = ""
        except errors.DuplicateKeyError:
            print("Username ou email já existe.")
        except Exception as e:
            print(f"Erro: {e}")

w_button.on_click(on_register)


In [ ]:

display(w_username, w_email, w_first, w_last, w_password, w_roles, w_button, w_out)


# Deletar Registro

In [ ]:
from pymongo import MongoClient
from urllib.parse import quote_plus
import os

user = os.environ["MONGODB_USER"]
pw = os.environ["MONGODB_PASSWORD"]
host = os.environ["MONGODB_HOST"]
db = os.environ["MONGODB_DATABASE"]
auth_collection = os.getenv("MONGODB_AUTH_COLLECTION")

uri = f"mongodb://{quote_plus(user)}:{quote_plus(pw)}@{host}/{db}"
client = MongoClient(uri)

USERNAME_TO_DELETE = ...

result = client[db][auth_collection].delete_many({"username": USERNAME_TO_DELETE})
print("Deletados:", result.deleted_count)
